In [163]:
import json
outs = json.load(open("male_concept_direct_prompt.json"))
outs2 = json.load(open("male_non_concept_direct_prompt.json"))
features_of_interest = {10:set(),30:set(),59:set()}

def print_stats(concept_dict, non_concept_dict, layer):
    top_indices = concept_dict[str(layer)]["top_indices"]
    top_indices_2 = non_concept_dict[str(layer)]["top_indices"]
    freq = {}
    for item in top_indices:
        for index in item:
            freq[index] = freq.get(index,0)+1

    freq2 = {}
    for item in top_indices_2:
        for index in item:
            freq2[index] = freq2.get(index,0)+1
    
    top_concept, top_non_concept = sorted(freq, key = freq.get, reverse=True)[:20],sorted(freq2, key = freq2.get, reverse=True)[:20]
    print("\nTop 20 feature comparision and respective frequencies left: concept, right: non concept\n")
    for idx1, idx2 in zip(top_concept, top_non_concept):
        print(f"{int(idx1)}: {freq[idx1]}", f"\t| {int(idx2)}: {freq2[idx2]}")

    print("\nfeatures in concept top 20 that are not in non concept non20:\n")
    feats = []
    for item in top_concept:
        if item not in top_non_concept:
            print(item, freq[item])
            feats.append(int(item))
    return feats
for layer in [10,30,59]:
    features_of_interest[layer].update(print_stats(outs, outs2, layer))



Top 20 feature comparision and respective frequencies left: concept, right: non concept

55540: 250 	| 40031: 250
40031: 250 	| 55540: 250
61659: 250 	| 61659: 250
72879: 250 	| 72879: 250
61131: 250 	| 61131: 250
64939: 250 	| 14327: 250
43391: 250 	| 64939: 250
14327: 250 	| 69487: 250
64706: 250 	| 43391: 250
69487: 249 	| 5494: 250
5494: 249 	| 64706: 250
50988: 225 	| 50988: 250
57520: 221 	| 57520: 250
13698: 139 	| 13698: 250
6023: 128 	| 6023: 250
57866: 125 	| 50323: 246
50323: 102 	| 57866: 220
5792: 70 	| 5792: 134
18075: 67 	| 71492: 131
44870: 44 	| 62924: 88

features in concept top 20 that are not in non concept non20:

18075.0 67
44870.0 44

Top 20 feature comparision and respective frequencies left: concept, right: non concept

21625: 250 	| 21625: 250
83220: 250 	| 22022: 250
51936: 250 	| 83220: 250
56791: 250 	| 51936: 250
22022: 249 	| 23806: 250
6189: 247 	| 80638: 250
26593: 247 	| 33513: 250
23806: 235 	| 44383: 250
80638: 225 	| 57720: 250
57720: 211 	| 42016:

In [158]:
def full_feature_set(concept_dict, non_concept_dict, layer):
    top_indices = concept_dict[str(layer)]["top_indices"]
    top_indices_2 = non_concept_dict[str(layer)]["top_indices"]
    freq = {}
    for item in top_indices:
        for index in item:
            freq[index] = freq.get(index,0)+1

    freq2 = {}
    for item in top_indices_2:
        for index in item:
            freq2[index] = freq2.get(index,0)+1
    return set(freq.keys())|set(freq2.keys())

def calculate_precision(feat, concept_dict, non_concept_dict, layer):
    concept_sum, non_concept_sum = 0,0
    for item, vals in zip(concept_dict[str(layer)]["top_indices"],concept_dict[str(layer)]["top_values"]):
        try:
            idx = item.index(feat)
            concept_sum += vals[idx]
            non_concept_sum += vals[idx]
        except ValueError:
            pass
    
    for item, vals in zip(non_concept_dict[str(layer)]["top_indices"],non_concept_dict[str(layer)]["top_values"]):
        try:
            idx = item.index(feat)
            non_concept_sum += vals[idx]
        except ValueError:
            pass
    return concept_sum/non_concept_sum
    
def freq_dicts(concept_dict, non_concept_dict, layer):
    top_indices = concept_dict[str(layer)]["top_indices"]
    top_indices_2 = non_concept_dict[str(layer)]["top_indices"]
    freq = {}
    for item in top_indices:
        for index in item:
            freq[index] = freq.get(index,0)+1

    freq2 = {}
    for item in top_indices_2:
        for index in item:
            freq2[index] = freq2.get(index,0)+1
    return freq,freq2

In [165]:
def specificity_feats(layer):
    all_feats = full_feature_set(outs, outs2, layer)
    precision = {}
    for feat in all_feats:
        precision[feat] = calculate_precision(feat, outs, outs2, layer)
    freq, freq2 = freq_dicts(outs, outs2, layer)
    l = sorted([(int(i),freq[i], precision[i]) for i in precision if i in freq],key=lambda x:(x[2],x[1]), reverse=True)
    for feat in l:
        if feat[2]>.7 and feat[1]>20:
            features_of_interest[layer].add(feat[0])
    print(l)

In [168]:
specificity_feats(30)

[(23389, 143, 1.0), (24026, 137, 1.0), (43399, 92, 1.0), (19441, 88, 1.0), (23272, 87, 1.0), (28532, 84, 1.0), (72702, 82, 1.0), (71976, 49, 1.0), (50004, 40, 1.0), (30365, 36, 1.0), (29532, 30, 1.0), (42156, 25, 1.0), (77186, 18, 1.0), (37971, 14, 1.0), (29840, 13, 1.0), (43258, 12, 1.0), (22175, 11, 1.0), (26030, 10, 1.0), (26809, 9, 1.0), (24304, 9, 1.0), (24313, 8, 1.0), (78434, 6, 1.0), (80994, 6, 1.0), (9256, 5, 1.0), (77533, 5, 1.0), (19686, 4, 1.0), (31528, 4, 1.0), (36142, 4, 1.0), (43994, 4, 1.0), (43041, 3, 1.0), (64628, 3, 1.0), (70888, 3, 1.0), (64889, 3, 1.0), (37242, 3, 1.0), (70041, 3, 1.0), (50138, 3, 1.0), (67576, 3, 1.0), (54288, 2, 1.0), (72747, 2, 1.0), (47152, 2, 1.0), (47155, 2, 1.0), (43594, 2, 1.0), (42061, 2, 1.0), (72811, 2, 1.0), (77432, 2, 1.0), (26743, 2, 1.0), (74888, 2, 1.0), (59047, 2, 1.0), (6322, 2, 1.0), (44727, 2, 1.0), (59083, 2, 1.0), (51919, 2, 1.0), (55001, 2, 1.0), (75002, 2, 1.0), (43266, 2, 1.0), (53530, 2, 1.0), (14627, 2, 1.0), (57654, 2, 1

In [169]:
features_of_interest

{10: {18075, 41517, 42622, 44870},
 30: {6189,
  19441,
  23272,
  23389,
  24026,
  28532,
  29532,
  30365,
  36153,
  42156,
  43399,
  50004,
  71976,
  72702},
 59: {5405, 65885, 83827}}

In [ ]:
# INDIRECT PROMPT
features_of_interest_10 = [15580, 41517, 15559, 34824, 44870, 18075]
features_of_interest_30 = [50367, 71976, 1074, 29840, 29532, 19441, 50004, 43399, 23389, 37971, 28532, 6189]
features_of_interest_59 = [63081, 23213, 5405, 65885, 49184, 45436, 40936, 48678, 50771]

# DIRECT PROMPT (ACTUALLY DONE)
features_of_interest_10 = [44870, 15559, 15580, 41517, 50976, 18075, 57820]
features_of_interest_30 = [50367, 24026, 19441, 23389, 43399, 28532, 71976, 72702, 29532, 23272, 77186, 22175, 80994, 50004, 42156, 37971, 29840,6189, 36153]
features_of_interest_59 = [21833, 5405, 50317, 49184, 65885, 40936, 83827] 

In [153]:
(34824, 37, 1.0)
(44870, 20, 1.0)
(15559, 45, 0.753808575509773)

(15559, 45, 0.753808575509773)

In [154]:
calculate_precision(50078.0, outs,outs2, 10)

0.4572430142751375

In [10]:
prompts = json.load(open("text_dataset/text_concept_a_female_person.json"))

In [31]:
vis = json.load(open("female_vis_direct_prompt.json"))

In [15]:
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained("google/gemma-3-27b-it")

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


In [137]:
from circuitsvis.tokens import colored_tokens_multi
import torch



def visualize(features, prompts,n_batches, batch_size, layer,chosen_concept, start_from=0, system_prompt_token_count=0):
    pad = [0.0]*len(features[str(layer)][0][0][0])
    batched = features[str(layer)][start_from].copy()
    for i in range(start_from+1,start_from+n_batches):
        batched += features[str(layer)][i].copy()

    max_len = len(max(batched, key=lambda x: len(x)))
    for item in batched:
        if len(item)<max_len:
            for _ in range(max_len-len(item)):
                item.insert(0,pad.copy())        

    messages = [[
                    {
                        "role": "system",
                        "content": [{"type": "text", "text": f"You are a linguistics expert, your task is to identify all words that fall under the linguistic umbrella of {chosen_concept}, whether that manifests in direct words, nouns, pronouns, etc"}]
                    },
                    {
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompts[i]}
                        ]
                    }
                ] for i in range(start_from*batch_size, n_batches*batch_size+start_from*batch_size)]
    # print(list(range(start_from*batch_size, n_batches*batch_size+start_from*batch_size)), list(range(start_from+1,start_from+n_batches)))
    tokens = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                    return_dict=True, return_tensors="pt", padding=True)
    str_tokens = [processor.decode(t) for t in tokens["input_ids"][:,system_prompt_token_count:].flatten(end_dim=1)]
    # Visualize activations for top 20 most prominent features
    return colored_tokens_multi(str_tokens, torch.tensor(batched, dtype=torch.float32)[:,system_prompt_token_count:,:].flatten(end_dim=1))



In [138]:
rendered = visualize(features=vis.copy(), prompts=prompts.copy(), n_batches=8, batch_size=2, layer=30,chosen_concept="A female person", start_from=0)

In [139]:
len(vis['30'][0][0][0])

19

In [140]:
messages = [
            {
                "role": "system",
                "content": [{"type": "text", "text": f"You are a linguistics expert, your task is to identify all words that fall under the linguistic umbrella of {chosen_concept}, whether that manifests in direct words, nouns, pronouns, etc"}]
            },
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": ""}
                ]
            }
        ]
tokens = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=True,
                    return_dict=True, return_tensors="pt", padding=True)

In [141]:
len(tokens['input_ids'][0])

46

In [142]:
tokens

{'input_ids': tensor([[     2,    105,   2364,    107,   3048,    659,    496, 127533,   7710,
         236764,    822,   4209,    563,    531,   8701,    784,   4171,    600,
           3798,   1208,    506,  45755,  36187,    529,    562,   8256,   1589,
         236764,   3363,    600,  99688,    528,   1982,   4171, 236764,  77093,
         236764, 107701, 236764,   4044,    108,    106,    107,    105,   4368,
            107]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])}

In [143]:
len([(i, processor.decode(j)) for i,j in enumerate(tokens["input_ids"][0][:-4])])

42

In [144]:
rendered

[[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]]